### Imports

In [ ]:
import pickle
import os
import sys
import torch
import numpy as np
import csv
import matplotlib.pyplot as plt
import pandas as pd
import tifffile
from torch.nn.functional import interpolate
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import matplotlib.cm as cm

In [ ]:
sys.path.append('/home/omer/git/AutoDS3D')

In [ ]:
from DS3Dplus.ds3d_utils import ImModel, Volume2XYZ

### Functions

In [ ]:
import copy

def _compute_overlay(im, xyzps, param_dict):
    if xyzps is None or im is None:
        print('One of the inputs is empty.')
        return None, None, None, None
    xyz_rec = xyzps[:, :3]
    H, W = im.shape

    # deep copy to avoid mutating the original param_dict
    pd_local = copy.deepcopy(param_dict)
    pd_local['H'], pd_local['W'] = H, W

    if H > pd_local['phase_mask'].shape[0] or W > pd_local['phase_mask'].shape[1]:
        sf = max(H // pd_local['phase_mask'].shape[0] + 1, W // pd_local['phase_mask'].shape[1] + 1)
        pd_local['ps_BFP'] /= sf
        phase_mask = pd_local['phase_mask']
        HW = np.floor(pd_local['f_4f'] * pd_local['lamda'] / (
                pd_local['ps_camera'] * pd_local['ps_BFP']))
        HW = int(HW + 1 - (HW % 2))

        phase_mask = interpolate(torch.tensor(phase_mask).unsqueeze(0).unsqueeze(1), size=(HW, HW))
        pd_local['phase_mask'] = phase_mask[0, 0].numpy()

    model = ImModel(pd_local)

    # nphotons_rec = (pd_local['Nsig_range'][0]+pd_local['Nsig_range'][1])/2 * np.ones(xyz_rec.shape[0])
    # psfs_rec = model.get_psfs(torch.from_numpy(np.c_[xyz_rec, nphotons_rec]).to(device)).cpu().numpy()
    im_norm = (im - im.min()) / (im.max() - im.min())

    ps_xy = pd_local['vs_xy'] * pd_local['us_factor']

    del model
    torch.cuda.empty_cache()

    return im_norm, xyz_rec, ps_xy


def select_strongest_localizations_by_radius(
    localizations,
    radius,
    x_col='x [nm]',
    y_col='y [nm]',
    intensity_col='intensity [au]',
):
    """Group nearby localizations in x/y and keep the strongest one per group."""
    if radius < 0:
        raise ValueError('radius must be non-negative.')

    if isinstance(localizations, pd.DataFrame):
        df = localizations.copy()
    else:
        df = pd.DataFrame(localizations)

    required_columns = [x_col, y_col, intensity_col]
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    if df.empty:
        return df.iloc[0:0].copy()

    from scipy.spatial import cKDTree

    coords = df[[x_col, y_col]].to_numpy(dtype=np.float64)
    tree = cKDTree(coords)
    visited = np.zeros(len(df), dtype=bool)
    selected_indices = []

    for start_idx in range(len(df)):
        if visited[start_idx]:
            continue

        component = []
        stack = [start_idx]
        visited[start_idx] = True

        while stack:
            current_idx = stack.pop()
            component.append(current_idx)
            neighbor_indices = tree.query_ball_point(coords[current_idx], radius)

            for neighbor_idx in neighbor_indices:
                if not visited[neighbor_idx]:
                    visited[neighbor_idx] = True
                    stack.append(neighbor_idx)

        component_df = df.iloc[component]
        strongest_index = component_df[intensity_col].astype(float).idxmax()
        selected_indices.append(strongest_index)

    return df.loc[selected_indices].sort_index().reset_index(drop=True)


def bias_comparison(param_dict, xyzps_baseline, im_baseline, xyzps_corrected, im_corrected, device):
    im_b, xyz_b, ps_xy = _compute_overlay(im_baseline, xyzps_baseline, param_dict)
    im_c, xyz_c, ps_xy = _compute_overlay(im_corrected, xyzps_corrected, param_dict)

    fig, axes = plt.subplots(1, 2, figsize=(10, 10))

    axes[0].imshow(im_b, cmap='gray')
    axes[0].plot(xyz_b[:, 0] / (ps_xy*1000), xyz_b[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[0].set_title('Baseline')
    axes[0].axis('off')

    axes[1].imshow(im_c, cmap='gray')
    axes[1].plot(xyz_c[:, 0] / (ps_xy*1000), xyz_c[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[1].set_title('Corrected')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def overlay_images_rg(im1: np.ndarray, im2: np.ndarray,
                      title: str = "",
                      figsize: tuple = (8, 8),
                      show: bool = True) -> np.ndarray:
    """Overlay two single-channel images with im1 in red and im2 in green.

    Each image is normalized independently to [0, 1] before compositing, so
    the dynamic range mismatch between channels does not hide structure.
    Pixels present in both images appear yellow; pixels unique to im1 appear
    red and pixels unique to im2 appear green.

    Args:
        im1: 2D array (H, W), shown as the red channel.
        im2: 2D array (H, W), shown as the green channel.
        title: Optional figure title.
        figsize: Matplotlib figure size in inches.
        show: If True, display the figure immediately.

    Returns:
        RGB overlay array of shape (H, W, 3), dtype float32, values in [0, 1].
    """
    def _norm(im):
        im = np.asarray(im, dtype=np.float32)
        lo, hi = im.min(), im.max()
        if hi > lo:
            return (im - lo) / (hi - lo)
        return np.zeros_like(im)

    r = _norm(im1)
    g = _norm(im2)
    rgb = np.stack([r, g, np.zeros_like(r)], axis=-1)

    if show:
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(rgb)
        if title:
            ax.set_title(title)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

    return rgb

In [ ]:

def match_localizations_hungarian(locs_a, locs_b, max_distance=None):
    """Match two sets of localizations with the Hungarian algorithm and return mean distance.

    Args:
        locs_a: (N, D) array of localizations
        locs_b: (M, D) array of localizations
        max_distance: optional distance cutoff; matched pairs beyond this are excluded.

    Returns:
        mean_dist: mean Euclidean distance of matched pairs.
        dists: (K,) array of individual matched distances after filtering.
        row_ind: (K,) indices into locs_a for each matched pair.
        col_ind: (K,) indices into locs_b for each matched pair.
    """
    C = cdist(locs_a, locs_b, metric='euclidean')
    row_ind, col_ind = linear_sum_assignment(C)
    dists = C[row_ind, col_ind]
    if max_distance is not None:
        keep = dists <= max_distance
        row_ind, col_ind, dists = row_ind[keep], col_ind[keep], dists[keep]
    return float(np.mean(dists)), dists, row_ind, col_ind


In [ ]:
def evaluate_jaccard_vs_radius(
    xyz_ref,
    xyz_det,
    optical_center,
    radius_bins,
    threshold_um,
    fov_size=None,
    title='',
    ps_xy=0.11,
    cmap='viridis',
):
    """Compute the cumulative Jaccard index per radial zone at a fixed threshold and plot it as a radial heat map.

    Zone b is cumulative — xyz_ref/xyz_det points with r < radius_bins[b] — exactly
    as before, just evaluated at one fixed matching threshold instead of a sweep.
    Only the rendering changes: each zone is drawn as the visual band (annulus)
    between radius_bins[b-1] and radius_bins[b] (innermost band is r < radius_bins[0]),
    filled with the color for that zone's cumulative Jaccard value — so the color of
    the outer band at radius r_b reflects the Jaccard considering all points within r_b.

    All spatial inputs (xyz_ref, xyz_det, optical_center, radius_bins) must be in
    **pixels** — the same unit used by match_localizations_hungarian. threshold_um
    is given in microns and converted internally to pixels via ps_xy.

    Args:
        xyz_ref: (N, 2) reference coordinates in pixels.
        xyz_det: (M, 2) detection coordinates in pixels.
        optical_center: (cx, cy) in pixels.
        radius_bins: 1-D array of outer-edge radii in pixels for each cumulative zone.
        threshold_um: matching distance threshold in microns.
        fov_size: (W, H) in pixels. Inferred from data if None.
        title: Optional suffix for the figure suptitle.
        ps_xy: Pixel size in μm/px (default 0.11 μm/px = 110 nm/px).
        cmap: Colormap used to encode each band's Jaccard value.
    Returns:
        fig: matplotlib Figure.
        jaccard_per_bin: (n_bins,) cumulative Jaccard index per zone at threshold_um.
        bin_radii: (n_bins,) outer-edge radii (pixels).
    """
    scale = ps_xy  # μm/px
    threshold_px = threshold_um / scale

    cx, cy = optical_center
    r_ref = np.sqrt((xyz_ref[:, 0] - cx)**2 + (xyz_ref[:, 1] - cy)**2)
    r_det = np.sqrt((xyz_det[:, 0] - cx)**2 + (xyz_det[:, 1] - cy)**2)

    bin_radii = np.asarray(radius_bins)
    n_bins = len(bin_radii)

    # Per-bin Jaccard at a fixed threshold, cumulative (r < r_hi) as before
    jaccard_per_bin = np.zeros(n_bins)
    for b, r_hi in enumerate(bin_radii):
        refs_b = xyz_ref[r_ref < r_hi]
        dets_b = xyz_det[r_det < r_hi]
        n_ref_b, n_det_b = len(refs_b), len(dets_b)

        if n_ref_b == 0 or n_det_b == 0:
            continue

        # match_localizations_hungarian returns distances in the same units as input (pixels)
        _, match_dists, _, _ = match_localizations_hungarian(dets_b, refs_b)
        tp = int(np.sum(match_dists < threshold_px))
        denom = n_det_b + n_ref_b - tp
        jaccard_per_bin[b] = tp / denom if denom > 0 else 0.0

    norm = plt.Normalize(0.0, 1.0)
    cmap_fn = plt.get_cmap(cmap)
    colors = cmap_fn(norm(jaccard_per_bin))

    if fov_size is None:
        all_pts = np.vstack([xyz_ref, xyz_det])
        fov_size = (int(all_pts[:, 0].max()) + 1, int(all_pts[:, 1].max()) + 1)
    W, H = fov_size

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    suptitle = f'Jaccard heat map (T={threshold_um:.2f} μm)—{title}' if title else f'Jaccard heat map (T={threshold_um:.2f} μm)'
    fig.suptitle(suptitle, fontsize=13)

    # Draw bands outer-to-inner: each smaller disc paints over the inner part of the
    # previous one, leaving a colored annulus per zone.
    for b in reversed(range(n_bins)):
        r_hi_um = bin_radii[b] * scale
        band = plt.Circle((0, 0), r_hi_um, facecolor=colors[b], edgecolor='k', lw=0.8, zorder=3)
        ax.add_patch(band)

    # Scatter reference points shifted to optical-center origin, in μm
    n_show = min(300, len(xyz_ref))
    idx = np.random.default_rng(0).choice(len(xyz_ref), n_show, replace=False)
    ax.scatter((xyz_ref[idx, 0] - cx) * scale,
               (xyz_ref[idx, 1] - cy) * scale,
               s=3, c='white', edgecolors='k', linewidths=0.2, zorder=5)

    # Optical center at origin
    ax.scatter(0, 0, c='red', s=120, marker='+', zorder=6, linewidths=2.5)

    # Annotate each band's cumulative Jaccard value near its outer edge
    for b in range(n_bins):
        r_hi_um = bin_radii[b] * scale
        ax.annotate(f'J={jaccard_per_bin[b]:.2f}', xy=(0, r_hi_um),
                    xytext=(2, 2), textcoords='offset points', fontsize=8, zorder=7)

    sm = cm.ScalarMappable(norm=norm, cmap=cmap_fn)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Jaccard index')

    # Grid lines through optical center
    ax.axhline(0, color='k', lw=0.5, zorder=2)
    ax.axvline(0, color='k', lw=0.5, zorder=2)

    # Axis limits in μm, centred at optical center; y inverted to match image convention
    ax.set_xlim(-cx * scale, (W - cx) * scale)
    ax.set_ylim((H - cy) * scale, -cy * scale)
    ax.set_aspect('equal')
    ax.set_xlabel('X (μm)')
    ax.set_ylabel('Y (μm)')
    ax.set_title('Radial zones around optical center')
    ax.grid(True, ls='--', alpha=0.3)

    plt.tight_layout()
    plt.show()

    return fig, jaccard_per_bin, bin_radii


In [ ]:
def _format_z_range(z_range):
    """Format a Z range key (scalar nm depth, or (low, high) nm pair) as a μm label."""
    if isinstance(z_range, (tuple, list, np.ndarray)) and len(z_range) == 2:
        lo, hi = z_range
        return f'Z: {lo / 1000:.2f}–{hi / 1000:.2f} μm'
    return f'Z = {z_range / 1000:.2f} μm'


def plot_jaccard_vs_radius_by_depth(
    z_ranges,
    jaccard_baseline_by_range,
    jaccard_corrected_by_range,
    optical_center,
    radius_bins,
    threshold_um,
    fov_size,
    cmap='viridis',
    figsize_per_row=2.6,
):
    """Grid of Jaccard-vs-radius radial heat maps, one row per Z range, baseline vs corrected side by side.

    Layout is len(z_ranges) rows x 3 columns: column 0 is a text label naming
    the row's Z range, column 1 is the baseline radial heat map, and column 2
    is the corrected one. Each heat map renders one colored annulus per entry
    of radius_bins — exactly the band rendering used in
    evaluate_jaccard_vs_radius — filled with that zone's cumulative Jaccard
    value (0-1) via cmap. All heat maps share one colorbar and axis scale so
    they're directly comparable across rows and between baseline/corrected.

    Args:
        z_ranges: Sequence of Z range keys, one per row (scalars or (low, high)
            pairs, in nm). Used both to look up the two dicts below and to
            label column 0.
        jaccard_baseline_by_range: dict mapping each entry of z_ranges to a
            1-D array of cumulative baseline Jaccard values, aligned 1:1 with
            radius_bins (same shape as jaccard_per_bin from evaluate_jaccard_vs_radius).
        jaccard_corrected_by_range: dict mapping each entry of z_ranges to a
            1-D array of cumulative corrected Jaccard values, aligned 1:1 with radius_bins.
        optical_center: (cx, cy) in pixels, shown in the figure suptitle for reference.
        radius_bins: 1-D array of outer-edge radii (μm) for each cumulative zone,
            shared by every row.
        threshold_um: Matching distance threshold (μm) used to compute the Jaccard
            values, shown in the figure suptitle.
        fov_size: (W, H) in pixels, shown in the figure suptitle for reference.
        cmap: Colormap used to encode each band's Jaccard value.
        figsize_per_row: Inches of figure height allotted per row.

    Returns:
        fig: matplotlib Figure with len(z_ranges) rows x 3 columns.
    """
    n_rows = len(z_ranges)
    radius_bins = np.asarray(radius_bins)
    fig, axes = plt.subplots(
        n_rows, 3, figsize=(9, figsize_per_row * n_rows),
        gridspec_kw={'width_ratios': [1, 3, 3]},
    )
    if n_rows == 1:
        axes = axes[None, :]

    norm = plt.Normalize(0.0, 1.0)
    cmap_fn = plt.get_cmap(cmap)
    r_max = float(radius_bins.max()) * 1.15

    def _draw_bands(ax, jaccard_per_bin):
        jaccard_per_bin = np.asarray(jaccard_per_bin)
        n_bins = min(len(radius_bins), len(jaccard_per_bin))
        colors = cmap_fn(norm(jaccard_per_bin[:n_bins]))
        for b in reversed(range(n_bins)):
            band = plt.Circle((0, 0), radius_bins[b], facecolor=colors[b],
                               edgecolor='k', lw=0.6, zorder=3)
            ax.add_patch(band)
        ax.scatter(0, 0, c='red', s=40, marker='+', zorder=6, linewidths=1.5)
        ax.set_xlim(-r_max, r_max)
        ax.set_ylim(-r_max, r_max)
        ax.set_aspect('equal')
        ax.grid(True, ls='--', alpha=0.3)

    cx, cy = optical_center
    W, H = fov_size
    fig.suptitle(
        f'Jaccard index vs radius by Z range (T={threshold_um:.2f} μm, '
        f'optical center=({cx:.0f}, {cy:.0f}) px, FOV={W}×{H} px)',
        fontsize=13,
    )

    for row, z_range in enumerate(z_ranges):
        ax_label, ax_base, ax_corr = axes[row]

        # --- column 0: Z range label ---
        ax_label.axis('off')
        ax_label.text(0.5, 0.5, _format_z_range(z_range), ha='center', va='center',
                       fontsize=11, transform=ax_label.transAxes)

        # --- column 1: baseline radial heat map ---
        _draw_bands(ax_base, jaccard_baseline_by_range[z_range])
        if row == 0:
            ax_base.set_title('Baseline')

        # --- column 2: corrected radial heat map ---
        _draw_bands(ax_corr, jaccard_corrected_by_range[z_range])
        if row == 0:
            ax_corr.set_title('Corrected')

    plt.tight_layout(rect=[0, 0, 0.92, 1])

    sm = cm.ScalarMappable(norm=norm, cmap=cmap_fn)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Jaccard index')

    plt.show()
    return fig

### Inputs

In [ ]:
param_dict_path = '/home/omer/data/mse3/test_of_axis/nfp_minus3p5_z_0_to_3p2/train_data_120x120/param.pickle'

In [ ]:
with open(param_dict_path, 'rb') as handle:
    param_dict = pickle.load(handle)

In [ ]:
device = 'cuda:2'
param_dict['device'] = device
localization_th = 40 # nm
param_dict['threshold'] = 10
ps_xy = param_dict['vs_xy'] * param_dict['us_factor']

In [ ]:
ps_xy

In [ ]:
model = ImModel(param_dict)
volume2xyz = Volume2XYZ(param_dict)

In [ ]:
baseline_dir = '/home/omer/data/mse3/train_data/baseline'
corrected_dir = '/home/omer/data/mse3/train_data/corrected'

In [ ]:
def evaluate_jaccard_vs_radius(
    xyz_ref,
    xyz_det,
    optical_center,
    radius_bins,
    threshold_um,
    fov_size=None,
    title='',
    ps_xy=0.11,
    cmap='viridis',
):
    """Compute the cumulative Jaccard index per radial zone at a fixed threshold and plot it as a radial heat map.

    Zone b is cumulative — xyz_ref/xyz_det points with r < radius_bins[b] — exactly
    as before, just evaluated at one fixed matching threshold instead of a sweep.
    Only the rendering changes: each zone is drawn as the visual band (annulus)
    between radius_bins[b-1] and radius_bins[b] (innermost band is r < radius_bins[0]),
    filled with the color for that zone's cumulative Jaccard value — so the color of
    the outer band at radius r_b reflects the Jaccard considering all points within r_b.

    All spatial inputs (xyz_ref, xyz_det, optical_center, radius_bins) must be in
    **pixels** — the same unit used by match_localizations_hungarian. threshold_um
    is given in microns and converted internally to pixels via ps_xy.

    Args:
        xyz_ref: (N, 2) reference coordinates in pixels.
        xyz_det: (M, 2) detection coordinates in pixels.
        optical_center: (cx, cy) in pixels.
        radius_bins: 1-D array of outer-edge radii in pixels for each cumulative zone.
        threshold_um: matching distance threshold in microns.
        fov_size: (W, H) in pixels. Inferred from data if None.
        title: Optional suffix for the figure suptitle.
        ps_xy: Pixel size in μm/px (default 0.11 μm/px = 110 nm/px).
        cmap: Colormap used to encode each band's Jaccard value.
    Returns:
        fig: matplotlib Figure.
        jaccard_per_bin: (n_bins,) cumulative Jaccard index per zone at threshold_um.
        bin_radii: (n_bins,) outer-edge radii (pixels).
    """
    scale = ps_xy  # μm/px
    threshold_px = threshold_um / scale

    cx, cy = optical_center
    r_ref = np.sqrt((xyz_ref[:, 0] - cx)**2 + (xyz_ref[:, 1] - cy)**2)
    r_det = np.sqrt((xyz_det[:, 0] - cx)**2 + (xyz_det[:, 1] - cy)**2)

    bin_radii = np.asarray(radius_bins)
    n_bins = len(bin_radii)

    # Per-bin Jaccard at a fixed threshold, cumulative (r < r_hi) as before
    jaccard_per_bin = np.zeros(n_bins)
    for b, r_hi in enumerate(bin_radii):
        refs_b = xyz_ref[r_ref < r_hi]
        dets_b = xyz_det[r_det < r_hi]
        n_ref_b, n_det_b = len(refs_b), len(dets_b)

        if n_ref_b == 0 or n_det_b == 0:
            continue

        # match_localizations_hungarian returns distances in the same units as input (pixels)
        _, match_dists, _, _ = match_localizations_hungarian(dets_b, refs_b)
        tp = int(np.sum(match_dists < threshold_px))
        denom = n_det_b + n_ref_b - tp
        jaccard_per_bin[b] = tp / denom if denom > 0 else 0.0

    norm = plt.Normalize(0.0, 1.0)
    cmap_fn = plt.get_cmap(cmap)
    colors = cmap_fn(norm(jaccard_per_bin))

    if fov_size is None:
        all_pts = np.vstack([xyz_ref, xyz_det])
        fov_size = (int(all_pts[:, 0].max()) + 1, int(all_pts[:, 1].max()) + 1)
    W, H = fov_size

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    suptitle = f'Jaccard heat map (T={threshold_um:.2f} μm)—{title}' if title else f'Jaccard heat map (T={threshold_um:.2f} μm)'
    fig.suptitle(suptitle, fontsize=13)

    # Draw bands outer-to-inner: each smaller disc paints over the inner part of the
    # previous one, leaving a colored annulus per zone.
    for b in reversed(range(n_bins)):
        r_hi_um = bin_radii[b] * scale
        band = plt.Circle((0, 0), r_hi_um, facecolor=colors[b], edgecolor='k', lw=0.8, zorder=3)
        ax.add_patch(band)

    # Scatter reference points shifted to optical-center origin, in μm
    n_show = min(300, len(xyz_ref))
    idx = np.random.default_rng(0).choice(len(xyz_ref), n_show, replace=False)
    ax.scatter((xyz_ref[idx, 0] - cx) * scale,
               (xyz_ref[idx, 1] - cy) * scale,
               s=3, c='white', edgecolors='k', linewidths=0.2, zorder=5)

    # Optical center at origin
    ax.scatter(0, 0, c='red', s=120, marker='+', zorder=6, linewidths=2.5)

    # Annotate each band's cumulative Jaccard value near its outer edge
    for b in range(n_bins):
        r_hi_um = bin_radii[b] * scale
        ax.annotate(f'J={jaccard_per_bin[b]:.2f}', xy=(0, r_hi_um),
                    xytext=(2, 2), textcoords='offset points', fontsize=8, zorder=7)

    sm = cm.ScalarMappable(norm=norm, cmap=cmap_fn)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Jaccard index')

    # Grid lines through optical center
    ax.axhline(0, color='k', lw=0.5, zorder=2)
    ax.axvline(0, color='k', lw=0.5, zorder=2)

    # Axis limits in μm, centred at optical center; y inverted to match image convention
    ax.set_xlim(-cx * scale, (W - cx) * scale)
    ax.set_ylim((H - cy) * scale, -cy * scale)
    ax.set_aspect('equal')
    ax.set_xlabel('X (μm)')
    ax.set_ylabel('Y (μm)')
    ax.set_title('Radial zones around optical center')
    ax.grid(True, ls='--', alpha=0.3)

    plt.tight_layout()
    plt.show()

    return fig, jaccard_per_bin, bin_radii


In [ ]:
ji_baseline_all_z_planes = []
ji_corrected_all_z_planes = []

radi_baseline_all_z_planes = []
radi_corrected_all_z_planes = []

for frame_depth, frame in zip([3200, 2400,1600,800,0],['frame_0015', 'frame_0020', 'frame_0025', 'frame_0030', 'frame_0035']):
    print(f'Processing frame {frame} at depth {frame_depth} nm...')

    # 4f system
    frame_loc_csv_path_4f = f"/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/4f_refrense/at4f_001_range10001_{frame}.csv"
    im_4f_path = f"/home/omer/experiments/mse3/at4f_001_range10001_frames/at4f_001_range10001_{frame}.tif"

    im_4f = tifffile.imread(im_4f_path)
    df_4f = pd.read_csv(frame_loc_csv_path_4f)
    df_filtered_4f = df_4f[(df_4f['z [nm]']>=(frame_depth - localization_th))&(df_4f['z [nm]']<=(frame_depth + localization_th))&(df_4f['intensity [au]']>=param_dict['threshold'])]
    df_filtered_4f.count()
    xyzps_4f = df_filtered_4f.to_numpy()[:, :4]
    z_errors_4f = np.abs(xyzps_4f[:, 2] - frame_depth)

    frame_loc_csv_path = f"/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/photon_count_alignment/orig/{frame}.csv"
    image_path = f"/home/omer/experiments/mse3/BehindObj_001_range10_frames/{frame}.tif"

    im_baseline = tifffile.imread(image_path) 

    df_baseline = pd.read_csv(frame_loc_csv_path)
    df_filtered_baseline = df_baseline[(df_baseline['z [nm]']>=(frame_depth - localization_th))&(df_baseline['z [nm]']<=(frame_depth + localization_th))&(df_baseline['intensity [au]']>=param_dict['threshold'])]
    df_filtered_baseline.count()
    xyzps_baseline = df_filtered_baseline.to_numpy()[:, :4]
    z_errors_baseline = np.mean(np.abs(xyzps_baseline[:, 2] - frame_depth))

    # corrected data
    corrected_image_path = f"/home/omer/training_results/mse3/beads/train_data_1200x1200/SFEnet_field_coord/2026-06-14_13-14-37/output/experiments/tiled_120/{frame}.tif"
    corrected_frame_loc_csv_path = f"/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/field_coord/doda/{frame}.csv"

    im_corrected = tifffile.imread(corrected_image_path)

    df = pd.read_csv(corrected_frame_loc_csv_path)
    df_filtered = df[(df['z [nm]']>=(frame_depth - localization_th))&(df['z [nm]']<=(frame_depth + localization_th))&(df['intensity [au]']>=param_dict['threshold'])]
    df_filtered.count()
    xyzps = df_filtered.to_numpy()[:, :4]
    z_errors_corrected = np.mean(np.abs(xyzps[:, 2] - frame_depth))

    xyzps_4f_pixels = xyzps_4f[:, :2] / (ps_xy*1000)
    xyzps_baseline_pixels = xyzps_baseline[:, :2] / (ps_xy*1000)
    xyzps_corrected_pixels = xyzps[:, :2] / (ps_xy*1000)

    xyzps_4f_pixels[:, 0] -= 6
    xyzps_4f_pixels[:, 1] += 15


    avg_distance_4f_baseline, distances1, row_ind1, col_ind1 = match_localizations_hungarian(xyzps_4f_pixels, xyzps_baseline_pixels, max_distance=50)
    avg_distance_4f_corrected, distances2, row_ind2, col_ind2 = match_localizations_hungarian(xyzps_4f_pixels, xyzps_corrected_pixels, max_distance=50)

    dist_to_center_micron1 = np.sqrt((xyzps_baseline_pixels[col_ind1, 0] - 647)**2 + (xyzps_baseline_pixels[col_ind1, 1] - 561)**2)*ps_xy
    dist_to_center_micron2 = np.sqrt((xyzps_corrected_pixels[col_ind2, 0] - 647)**2 + (xyzps_corrected_pixels[col_ind2, 1] - 561)**2)*ps_xy


    fig, axes = plt.subplots(1, 3, figsize=(10, 5))
    axes[0].imshow(im_baseline, cmap='gray')
    axes[0].plot(xyzps_baseline[:, 0] / (ps_xy*1000), xyzps_baseline[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[0].set_title('Baseline')
    axes[0].axis('off')
    axes[1].imshow(im_corrected, cmap='gray')
    axes[1].plot(xyzps[:, 0] / (ps_xy*1000), xyzps[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[1].set_title('Corrected')
    axes[1].axis('off')
    axes[2].imshow(im_4f, cmap='gray')
    axes[2].plot(xyzps_4f[:, 0] / (ps_xy*1000), xyzps_4f[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[2].set_title('4f system')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

    fig,ax = plt.subplots(figsize=(8, 8))
    ax.scatter(647,561, s=12, c='black', label='Optical center', marker='+', zorder=4)
    ax.scatter(xyzps_4f_pixels[:, 0], xyzps_4f_pixels[:, 1], s=4, c='tab:green', label='4f system', zorder=3)
    ax.scatter(xyzps_baseline_pixels[:, 0], xyzps_baseline_pixels[:, 1], s=4, c='tab:red',  alpha=0.5, label='Baseline', zorder=3)
    ax.scatter(xyzps_corrected_pixels[:, 0], xyzps_corrected_pixels[:, 1], s=4, c='tab:blue', alpha=0.5, label='Corrected', zorder=3)
    ax.plot([],[],color='white', label=f'4f vs Baseline: {avg_distance_4f_baseline*(ps_xy*1000):.1f} nm', zorder=4) 
    ax.plot([],[],color='white', label=f'4f vs Corrected: {avg_distance_4f_corrected*(ps_xy*1000):.1f} nm', zorder=4) 
    ax.set_xlim(0, 1200)
    ax.set_ylim(0, 1200)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_xlabel(f'X (pixels)')
    ax.set_ylabel(f'Y (pixels)')
    ax.set_title(f'Frame {frame} at depth {frame_depth} nm')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # --- Jaccard index vs radius threshold ---
    #r_thresholds_px = np.linspace(0, 10, 100)
    r_max_px = max(np.max(distances1), np.max(distances2))
    r_thresholds_px = np.linspace(0, r_max_px, 100)
    r_thresholds_um = r_thresholds_px * ps_xy

    n_4f_j = len(xyzps_4f_pixels)
    n_baseline_j = len(xyzps_baseline_pixels)
    n_corrected_j = len(xyzps_corrected_pixels)

    def _jaccard_and_color(distances, dist_to_center, n_ref):
        jaccards, colors = [], []
        for r in r_thresholds_px:
            mask = distances <= r
            tp = int(np.sum(mask))
            denom = n_4f_j + n_ref - tp
            j = tp / denom if denom > 0 else 0.0
            c = float(np.mean(dist_to_center[mask])) if tp > 0 else np.nan
            jaccards.append(j)
            colors.append(c)
        return np.array(jaccards), np.array(colors)
    
    j_baseline, c_baseline = _jaccard_and_color(distances1, dist_to_center_micron1, n_baseline_j)
    j_corrected, c_corrected = _jaccard_and_color(distances2, dist_to_center_micron2, n_corrected_j)

    ji_baseline_all_z_planes.extend(j_baseline)
    ji_corrected_all_z_planes.extend(j_corrected)

    fig_j, ax_j = plt.subplots(1, 1, figsize=(13, 5))
    idx1 = j_baseline.argmax()
    idx2 = j_corrected.argmax()

    radi_baseline_all_z_planes.append(r_thresholds_um[:idx1+1])
    radi_corrected_all_z_planes.append(r_thresholds_um[:idx2+1])
    
    sc1 = ax_j.plot(r_thresholds_um[:idx1+1], j_baseline[:idx1+1],  label=f'Baseline, avg lateral error {avg_distance_4f_baseline*(ps_xy):.2f} μm')
    sc2 = ax_j.plot(r_thresholds_um[:idx2+1], j_corrected[:idx2+1], label=f'Corrected, avg lateral error {avg_distance_4f_corrected*(ps_xy):.2f} μm')

    ax_j.set_xlabel('Radius threshold (μm)')
    ax_j.set_ylabel('Jaccard index')
    ax_j.grid(True, linestyle='--', alpha=0.5)
    ax_j.legend()
    fig_j.suptitle(f'Jaccard index vs radius threshold — at Z =  {frame_depth/1000} μm', fontsize=11)
    plt.tight_layout()
    plt.show()

    optical_center_pixels = (647, 561)
    RADIUS_BINS = np.array([150, 300, 450, 600])
    THRESHOLD_UM = 0.5  # µm, fixed lateral matching threshold for the ring heatmap
    FOV_SIZE = (1200, 1200)
    fig, jaccard_per_bin, bin_radii = evaluate_jaccard_vs_radius(
        xyzps_4f_pixels, xyzps_baseline_pixels, optical_center_pixels, RADIUS_BINS,
        threshold_um=THRESHOLD_UM,
        fov_size=FOV_SIZE,
        title=f'at depth {frame_depth} nm baseline',
        )
    
    fig, jaccard_per_bin, bin_radii = evaluate_jaccard_vs_radius(
        xyzps_4f_pixels, xyzps_corrected_pixels, optical_center_pixels, RADIUS_BINS,
        threshold_um=THRESHOLD_UM,
        fov_size=FOV_SIZE,
        title=f'at depth {frame_depth} nm corrected',
        )
    print('Outer radii (px):       ', bin_radii)
    print('Jaccard per zone:       ', jaccard_per_bin.round(3))
   

    print(f"Frame {frame} at depth {frame_depth} nm:")
    print(f"Baseline: {len(xyzps_baseline)} localizations, mean Z error: {z_errors_baseline:.2f} nm")
    print(f"Corrected: {len(xyzps)} localizations, mean Z error: {z_errors_corrected:.2f} nm")
    print(f"4f system: {len(xyzps_4f)} localizations, mean Z error: {np.mean(np.abs(xyzps_4f[:, 2] - frame_depth)):.2f} nm")
    print(f"4f vs Baseline: {avg_distance_4f_baseline*(ps_xy*1000):.1f} nm")
    print(f"4f vs Corrected: {avg_distance_4f_corrected*(ps_xy*1000):.1f} nm")


In [ ]:
ji_baseline_all_z_planes = []
ji_corrected_all_z_planes = []

radi_baseline_all_z_planes = []
radi_corrected_all_z_planes = []

im_4f_dir = "/home/omer/experiments/mse3/16feg26_microtubules/4f_exc_640nm_oil_x100_145_007_frames/"
frame_loc_csv_path_4f_dir = '/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/16feb26_microtubules/4f_refrense'

baseline_images_dir = "/home/omer/experiments/mse3/16feg26_microtubules/behind_obj_exc_640nm_oil_x100_145_009_frames/"
baseline_frame_loc_csv_path = "/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/16feb26_microtubules/orig"

corrected_img_dir = "/home/omer/training_results/mse3/beads/train_data_1200x1200/SFEnet_field_coord/2026-06-14_13-14-37/output/experiments/microtubules_beads/tile_120/"
corrected_frame_loc_csv_path = "/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/16feb26_microtubules/doda"


for frame_depth, frame in zip([2400,1600,800],['0020', '0025', '0033']):
    print(f'Processing frame {frame} at depth {frame_depth} nm...')

    # 4f system
    frame_loc_csv_path_4f = f"{frame_loc_csv_path_4f_dir}/{frame}.csv"
    im_4f_path = f"{im_4f_dir}/{frame}.tif"

    im_4f = tifffile.imread(im_4f_path)
    df_4f = pd.read_csv(frame_loc_csv_path_4f)
    df_filtered_4f = df_4f[(df_4f['z [nm]']>=(frame_depth - localization_th))&(df_4f['z [nm]']<=(frame_depth + localization_th))&(df_4f['intensity [au]']>=param_dict['threshold'])]
    df_filtered_4f.count()
    xyzps_4f = df_filtered_4f.to_numpy()[:, :4]
    z_errors_4f = np.abs(xyzps_4f[:, 2] - frame_depth)

    frame_loc_csv_path = f"{baseline_frame_loc_csv_path}/{frame}.csv"
    image_path = f"{baseline_images_dir}/{frame}.tif"

    im_baseline = tifffile.imread(image_path) 

    df_baseline = pd.read_csv(frame_loc_csv_path)
    df_filtered_baseline = df_baseline[(df_baseline['z [nm]']>=(frame_depth - localization_th))&(df_baseline['z [nm]']<=(frame_depth + localization_th))&(df_baseline['intensity [au]']>=param_dict['threshold'])]
    df_filtered_baseline.count()
    xyzps_baseline = df_filtered_baseline.to_numpy()[:, :4]
    z_errors_baseline = np.mean(np.abs(xyzps_baseline[:, 2] - frame_depth))

    # corrected data
    corrected_image_path = f"{corrected_img_dir}/{frame}.tif"
    corrected_frame_loc_csv_path_full = f"{corrected_frame_loc_csv_path}/{frame}.csv"

    im_corrected = tifffile.imread(corrected_image_path)

    df = pd.read_csv(corrected_frame_loc_csv_path_full)
    df_filtered = df[(df['z [nm]']>=(frame_depth - localization_th))&(df['z [nm]']<=(frame_depth + localization_th))&(df['intensity [au]']>=param_dict['threshold'])]
    df_filtered.count()
    xyzps = df_filtered.to_numpy()[:, :4]
    z_errors_corrected = np.mean(np.abs(xyzps[:, 2] - frame_depth))

    xyzps_4f_pixels = xyzps_4f[:, :2] / (ps_xy*1000)
    xyzps_baseline_pixels = xyzps_baseline[:, :2] / (ps_xy*1000)
    xyzps_corrected_pixels = xyzps[:, :2] / (ps_xy*1000)

    #xyzps_4f_pixels[:, 0] -= 6
    #xyzps_4f_pixels[:, 1] += 15


    avg_distance_4f_baseline, distances1, row_ind1, col_ind1 = match_localizations_hungarian(xyzps_4f_pixels, xyzps_baseline_pixels, max_distance=50)
    avg_distance_4f_corrected, distances2, row_ind2, col_ind2 = match_localizations_hungarian(xyzps_4f_pixels, xyzps_corrected_pixels, max_distance=50)

    dist_to_center_micron1 = np.sqrt((xyzps_baseline_pixels[col_ind1, 0] - 647)**2 + (xyzps_baseline_pixels[col_ind1, 1] - 561)**2)*ps_xy
    dist_to_center_micron2 = np.sqrt((xyzps_corrected_pixels[col_ind2, 0] - 647)**2 + (xyzps_corrected_pixels[col_ind2, 1] - 561)**2)*ps_xy


    fig, axes = plt.subplots(1, 3, figsize=(10, 5))
    axes[0].imshow(im_baseline, cmap='gray')
    axes[0].plot(xyzps_baseline[:, 0] / (ps_xy*1000), xyzps_baseline[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[0].set_title('Baseline')
    axes[0].axis('off')
    axes[1].imshow(im_corrected, cmap='gray')
    axes[1].plot(xyzps[:, 0] / (ps_xy*1000), xyzps[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[1].set_title('Corrected')
    axes[1].axis('off')
    axes[2].imshow(im_4f, cmap='gray')
    axes[2].plot(xyzps_4f[:, 0] / (ps_xy*1000), xyzps_4f[:, 1] / (ps_xy*1000), 'r.', markersize=1)
    axes[2].set_title('4f system')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

    fig,ax = plt.subplots(figsize=(8, 8))
    ax.scatter(647,561, s=12, c='black', label='Optical center', marker='+', zorder=4)
    ax.scatter(xyzps_4f_pixels[:, 0], xyzps_4f_pixels[:, 1], s=4, c='tab:green', label='4f system', zorder=3)
    ax.scatter(xyzps_baseline_pixels[:, 0], xyzps_baseline_pixels[:, 1], s=4, c='tab:red',  alpha=0.5, label='Baseline', zorder=3)
    ax.scatter(xyzps_corrected_pixels[:, 0], xyzps_corrected_pixels[:, 1], s=4, c='tab:blue', alpha=0.5, label='Corrected', zorder=3)
    ax.plot([],[],color='white', label=f'4f vs Baseline: {avg_distance_4f_baseline*(ps_xy*1000):.1f} nm', zorder=4) 
    ax.plot([],[],color='white', label=f'4f vs Corrected: {avg_distance_4f_corrected*(ps_xy*1000):.1f} nm', zorder=4) 
    ax.set_xlim(0, 1200)
    ax.set_ylim(0, 1200)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_xlabel(f'X (pixels)')
    ax.set_ylabel(f'Y (pixels)')
    ax.set_title(f'Frame {frame} at depth {frame_depth} nm')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # --- Jaccard index vs radius threshold ---
    #r_thresholds_px = np.linspace(0, 10, 100)
    r_max_px = max(np.max(distances1), np.max(distances2))
    r_thresholds_px = np.linspace(0, r_max_px, 100)
    r_thresholds_um = r_thresholds_px * ps_xy

    n_4f_j = len(xyzps_4f_pixels)
    n_baseline_j = len(xyzps_baseline_pixels)
    n_corrected_j = len(xyzps_corrected_pixels)

    def _jaccard_and_color(distances, dist_to_center, n_ref):
        jaccards, colors = [], []
        for r in r_thresholds_px:
            mask = distances <= r
            tp = int(np.sum(mask))
            denom = n_4f_j + n_ref - tp
            j = tp / denom if denom > 0 else 0.0
            c = float(np.mean(dist_to_center[mask])) if tp > 0 else np.nan
            jaccards.append(j)
            colors.append(c)
        return np.array(jaccards), np.array(colors)
    
    j_baseline, c_baseline = _jaccard_and_color(distances1, dist_to_center_micron1, n_baseline_j)
    j_corrected, c_corrected = _jaccard_and_color(distances2, dist_to_center_micron2, n_corrected_j)

    ji_baseline_all_z_planes.extend(j_baseline)
    ji_corrected_all_z_planes.extend(j_corrected)

    fig_j, ax_j = plt.subplots(1, 1, figsize=(13, 5))
    idx1 = j_baseline.argmax()
    idx2 = j_corrected.argmax()

    radi_baseline_all_z_planes.append(r_thresholds_um[:idx1+1])
    radi_corrected_all_z_planes.append(r_thresholds_um[:idx2+1])
    
    sc1 = ax_j.plot(r_thresholds_um[:idx1+1], j_baseline[:idx1+1],  label=f'Baseline, avg lateral error {avg_distance_4f_baseline*(ps_xy):.2f} μm')
    sc2 = ax_j.plot(r_thresholds_um[:idx2+1], j_corrected[:idx2+1], label=f'Corrected, avg lateral error {avg_distance_4f_corrected*(ps_xy):.2f} μm')

    ax_j.set_xlabel('Radius threshold (μm)')
    ax_j.set_ylabel('Jaccard index')
    ax_j.grid(True, linestyle='--', alpha=0.5)
    ax_j.legend()
    fig_j.suptitle(f'Jaccard index vs radius threshold — at Z =  {frame_depth/1000} μm', fontsize=11)
    plt.tight_layout()
    plt.show()

    optical_center_pixels = (504, 672)
    RADIUS_BINS = np.array([150, 300, 450, 600])
    THRESHOLD_UM = 0.5  # µm, fixed lateral matching threshold for the ring heatmap
    FOV_SIZE = (1200, 1200)
    fig, jaccard_per_bin, bin_radii = evaluate_jaccard_vs_radius(
        xyzps_4f_pixels, xyzps_baseline_pixels, optical_center_pixels, RADIUS_BINS,
        threshold_um=THRESHOLD_UM,
        fov_size=FOV_SIZE,
        title=f'at depth {frame_depth} nm baseline',
        )
    
    fig, jaccard_per_bin, bin_radii = evaluate_jaccard_vs_radius(
        xyzps_4f_pixels, xyzps_corrected_pixels, optical_center_pixels, RADIUS_BINS,
        threshold_um=THRESHOLD_UM,
        fov_size=FOV_SIZE,
        title=f'at depth {frame_depth} nm corrected',
        )
    print('Outer radii (px):       ', bin_radii)
    print('Jaccard per zone:       ', jaccard_per_bin.round(3))
   

    print(f"Frame {frame} at depth {frame_depth} nm:")
    print(f"Baseline: {len(xyzps_baseline)} localizations, mean Z error: {z_errors_baseline:.2f} nm")
    print(f"Corrected: {len(xyzps)} localizations, mean Z error: {z_errors_corrected:.2f} nm")
    print(f"4f system: {len(xyzps_4f)} localizations, mean Z error: {np.mean(np.abs(xyzps_4f[:, 2] - frame_depth)):.2f} nm")
    print(f"4f vs Baseline: {avg_distance_4f_baseline*(ps_xy*1000):.1f} nm")
    print(f"4f vs Corrected: {avg_distance_4f_corrected*(ps_xy*1000):.1f} nm")

In [ ]:
r_thresholds_um = [15,30,45,60]
radi_baseline_all_z_planes = [[0.45,0.16,0.10,0.09],[0.6,0.35,0.21,0.18],[1,0.56,0.5,0.45],[1.0,0.71,0.76,0.65],[0.8,0.93,0.73,0.71]]
radi_corrected_all_z_planes = [[0.5,0.32,0.41,0.41],[1,0.71,0.7,0.73],[1,1,0.8,0.78],[1.0,0.87,0.61,0.56],[0.5,0.42,0.3,0.28]]
jb_dict = {f'Z={z} nm': j for z, j in zip([3200, 2400,1600,800,0], radi_baseline_all_z_planes)}
jc_dict = {f'Z={z} nm': j for z, j in zip([3200, 2400,1600,800,0], radi_corrected_all_z_planes)}

In [ ]:
fig = plot_jaccard_vs_radius_by_depth(
    z_ranges=[3200, 2400, 1600, 800, 0],   # or list of (lo, hi) nm tuples
    jaccard_baseline_by_range=jb_dict,      # {z_range: jaccard array}
    jaccard_corrected_by_range=jc_dict,     # {z_range: jaccard array}
    optical_center=(600, 600),
    radius_bins=r_thresholds_um,
    threshold_um=0.5,
    fov_size=(1200, 1200),
)


In [ ]:
df_orig_all = pd.DataFrame()
df_corrected_all = pd.DataFrame()
# XY correction as a function of Z
for frame_depth, frame in zip([3200, 2400,1600,800,0],['frame_0015', 'frame_0020', 'frame_0025', 'frame_0030', 'frame_0035']):
    ## original data
    frame_loc_csv_path = f"/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/photon_count_alignment/orig/{frame}.csv"
    # corrected data
    corrected_frame_loc_csv_path = f"/home/omer/git/AutoDS3D/train_results/mse3/nfp_minus_3p5_z_0_to_3p2/field_coord/doda/{frame}.csv"

    df_orig = pd.read_csv(frame_loc_csv_path)
    df_orig = df_orig[(df_orig['z [nm]']>=(frame_depth - localization_th))&(df_orig['z [nm]']<=(frame_depth + localization_th))&(df_orig['intensity [au]']>=param_dict['threshold'])]
    df_orig_all = pd.concat([df_orig_all, df_orig], ignore_index=True)

    df_corrected = pd.read_csv(corrected_frame_loc_csv_path)
    df_corrected = df_corrected[(df_corrected['z [nm]']>=(frame_depth - localization_th))&(df_corrected['z [nm]']<=(frame_depth + localization_th))&(df_corrected['intensity [au]']>=param_dict['threshold'])]
    
    df_corrected_all = pd.concat([df_corrected_all, df_corrected], ignore_index=True)

xyzps_orig = df_orig_all.to_numpy()[:, :4]
xyzps_orig[:,0] = xyzps_orig[:,0] / (ps_xy*1000)
xyzps_orig[:,1] = xyzps_orig[:,1] / (ps_xy*1000)

xyzps_corrected = df_corrected_all.to_numpy()[:, :4]
xyzps_corrected[:,0] = xyzps_corrected[:,0] / (ps_xy*1000)
xyzps_corrected[:,1] = xyzps_corrected[:,1] / (ps_xy*1000)

fig, axes = plt.subplots(1, 2, figsize=(10, 10))
axes[0].scatter(xyzps_orig[:, 0], xyzps_orig[:, 1], s=1, c='tab:red', label=f'Baseline')
axes[0].set_title('Baseline')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')
axes[0].set_xlim(0, 1200)
axes[0].set_ylim(0, 1200)
axes[0].scatter(647,561, s=1, c='cyan', label='Reference point', edgecolors='k', marker='X')
axes[0].invert_yaxis()
axes[1].scatter(xyzps_corrected[:, 0], xyzps_corrected[:, 1], s=1, c='tab:blue', label='Corrected')
axes[1].set_title('Corrected')
axes[1].set_xlabel('X (pixels)')
axes[1].set_ylabel('Y (pixels)')
axes[1].set_xlim(0, 1200)
axes[1].set_ylim(0, 1200)
axes[1].scatter(647,561, s=1, c='cyan', label='Reference point', edgecolors='k', marker='X')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Release GPU memory
if 'model' in locals():
    del model
if 'volume2xyz' in locals():
    del volume2xyz
import gc
torch.cuda.empty_cache()
gc.collect()